In [ ]:
# ============================================================
# 环境配置
# - Colab 用户：取消注释下方 Colab 区块
# - 本地 Jupyter 用户：直接运行 Local 区块
# ============================================================

# ── Colab 环境（取消注释后运行） ──
# !pip install torch torchvision transformers matplotlib numpy pillow -q

# ── 本地 Jupyter 环境 ──
import subprocess
import sys

def _install(pkg: str):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for package in ["torch", "torchvision", "transformers", "matplotlib", "numpy", "pillow"]:
    _install(package)

# CLIP 代码实战：学习实现 vs 工程实现

基于论文 *Learning Transferable Visual Models From Natural Language Supervision*（Radford et al., 2021），
用 **CIFAR-10 零样本分类** 任务演示 CLIP 的核心思想：**把图像和文本对齐到同一个向量空间**。

本 Notebook 保留两条并行路径，并且使用 **同一组类别标签、同一份评估数据**：

| | 学习路径 (Learning) | 工程路径 (Engineering) |
|---|---|---|
| 目标 | 理解 CLIP 的关键计算模块 | 掌握工业级预训练 CLIP 的使用方式 |
| 实现方式 | **L2：关键模块演示**（Tiny image encoder + Tiny text encoder + 对比损失） | **E1：成熟库 + inference-only**（`transformers` + `openai/clip-vit-base-patch32`） |
| 代码量 | 较多，显式展示 shape 与损失 | 较少，直接调用处理器与预训练模型 |
| 适合场景 | 面试准备、原理拆解、理解 prompt / temperature / symmetric loss | 工程验证、零样本分类、批量推理、prompt ensembling |

> 两条路径不是两套无关代码，而是同一套 CLIP 思想在不同抽象层级上的表达。

In [ ]:
import random
from typing import List

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch import nn
from torch.utils.data import DataLoader, Subset
from transformers import AutoProcessor, CLIPModel

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## Section i：论文背景、任务定义与范围

### 1. 论文与任务
- **论文**：*Learning Transferable Visual Models From Natural Language Supervision*（OpenAI, 2021）
- **核心任务**：给定图像与文本，学习一个共享嵌入空间，使正确图文对相似度更高、错误配对更低。
- **下游演示任务**：CIFAR-10 零样本分类。做法不是训练一个 10 类 softmax 分类头，而是把 10 个类别写成 10 条文本提示，再比较图像和文本的相似度。

### 2. 为什么 CIFAR-10 适合这份 Notebook
- 数据集可直接下载，适合 Colab / 本地 Jupyter。
- 类别短、模板清晰，便于演示 prompt template。
- 32×32 小图适合学习路径快速跑通。

### 3. 本 Notebook 覆盖与不覆盖的边界
**覆盖：**
- 图像编码器 / 文本编码器 / 投射层 / L2 归一化 / 温度参数 / 对比损失 / 零样本推理
- prompt template 与 prompt ensembling 的工程使用

**不覆盖：**
- OpenAI 原论文中的大规模 4 亿图文对训练细节
- 大型 ViT / ResNet 主干的完整复现
- 大规模检索、分布式训练、长文本 caption 训练

## Section ii：最小必要理论

CLIP 可以压缩成下面 4 个公式：

### 1. 图像 / 文本编码
给定一批图像 $I$ 与文本 $T$：

$$I_f = f_{image}(I), \qquad T_f = f_{text}(T)$$

### 2. 投射到共享空间并归一化

$$I_e = \frac{W_i I_f}{\|W_i I_f\|_2}, \qquad T_e = \frac{W_t T_f}{\|W_t T_f\|_2}$$

### 3. 相似度矩阵与温度参数

$$\text{logits} = \exp(s) \cdot I_e T_e^T$$

其中 $s$ 是可学习的 `logit_scale`，等价于温度参数的对数形式。温度越高，softmax 分布越尖锐。

### 4. 对称对比损失（symmetric contrastive loss）

$$\mathcal{L} = \frac{1}{2}\big(\text{CE}(\text{logits}, y) + \text{CE}(\text{logits}^T, y)\big)$$

这里 $y = [0,1,2,\dots,N-1]$，表示一个 batch 内第 $i$ 张图像应当与第 $i$ 条文本配对。

> 这份 Notebook 只保留理解代码所需的最小理论；更完整的论文细节请回看配套 `01_CLIP.md`。

### 组件映射表（Mandatory）

| 论文组件 | 学习路径覆盖 | 工程库 / API 对应 | 状态 |
|---|---|---|---|
| 图像编码器 | `TinyImageEncoder`：小型 CNN + projector | `CLIPModel.get_image_features()` | Runnable |
| 文本编码器 | `TinyTextEncoder`：Embedding + self-attention + projector | `CLIPModel.get_text_features()` | Runnable |
| 线性投射层 | 在 image / text encoder 末端显式实现 | CLIP 内部 projection layers | Runnable |
| L2 归一化 | `F.normalize(...)` 显式实现 | CLIP 输出特征后同样需要归一化 | Runnable |
| 温度参数 `logit_scale` | `nn.Parameter(log(1/0.07))` | `model.logit_scale.exp()` | Runnable |
| 图文相似度矩阵 | `image_features @ text_features.T` | `outputs.logits_per_image` / 手工点积 | Runnable |
| 对称对比损失 | 手写双向 CE | 预训练阶段内部使用；推理阶段不再训练 | Runnable |
| Prompt template 分类器 | `a photo of a {class}` | `AutoProcessor(text=..., images=...)` | Runnable |
| Prompt ensembling | 多模板平均文本特征 | 多次 `get_text_features()` 后取平均 | Runnable |
| 原论文 4 亿图文对训练 | 只做小规模教学演示 | 不在本 Notebook 中复现 | Explain-only |

## Section 1：数据准备

学习路径与工程路径都使用 CIFAR-10 的同一组类别名称：
`airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck`。

为了让 Notebook 在 CPU / 免费 Colab 上稳定运行：
- 学习路径：从训练集抽一个 **平衡小子集** 做教学演示。
- 工程路径：从测试集抽一个 **平衡评估子集** 做零样本对比。

同时，这里固定基础模板：`a photo of a {class}`，后面再扩展到 prompt ensembling。

In [ ]:
CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

BASE_TEMPLATE = "a photo of a {}"
ENSEMBLE_TEMPLATES = [
    "a photo of a {}",
    "a blurry photo of a {}",
    "a bright photo of a {}",
    "a close-up photo of a {}",
]

train_transform = T.Compose([
    T.Resize((32, 32)),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

eval_transform = T.Compose([
    T.Resize((32, 32)),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_tensor_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=train_transform
)
test_tensor_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=eval_transform
)
test_raw_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=None
)

def balanced_indices(targets, per_class, seed=42):
    rng = np.random.default_rng(seed)
    targets = np.array(targets)
    picked = []
    for class_id in range(10):
        class_indices = np.where(targets == class_id)[0]
        chosen = rng.choice(class_indices, size=per_class, replace=False)
        picked.extend(chosen.tolist())
    return sorted(picked)

TRAIN_PER_CLASS = 80
TEST_PER_CLASS = 20

train_indices = balanced_indices(train_tensor_dataset.targets, TRAIN_PER_CLASS, seed=42)
test_indices = balanced_indices(test_tensor_dataset.targets, TEST_PER_CLASS, seed=123)

learning_train_subset = Subset(train_tensor_dataset, train_indices)
learning_test_subset = Subset(test_tensor_dataset, test_indices)
engineering_test_subset = Subset(test_raw_dataset, test_indices)

learning_train_loader = DataLoader(learning_train_subset, batch_size=64, shuffle=True, num_workers=0)
learning_test_loader = DataLoader(learning_test_subset, batch_size=64, shuffle=False, num_workers=0)

print(f"Learning train subset: {len(learning_train_subset)}")
print(f"Shared eval subset:    {len(learning_test_subset)}")

In [ ]:
sample_images, sample_labels = next(iter(learning_train_loader))
print(f"Image batch shape: {tuple(sample_images.shape)}")
print(f"Label batch shape: {tuple(sample_labels.shape)}")
print("Example labels:", [CIFAR10_CLASSES[i] for i in sample_labels[:8].tolist()])

In [ ]:
# ── Shared hyperparameters / 超参数集中管理 ──
D_MODEL = 128
NUM_HEADS = 4
D_FF = 256
MAX_TEXT_LEN = 8
LEARNING_RATE = 5e-4
NUM_EPOCHS = 4
ENGINEERING_BATCH_SIZE = 32

print({
    "D_MODEL": D_MODEL,
    "NUM_HEADS": NUM_HEADS,
    "D_FF": D_FF,
    "MAX_TEXT_LEN": MAX_TEXT_LEN,
    "LEARNING_RATE": LEARNING_RATE,
    "NUM_EPOCHS": NUM_EPOCHS,
    "ENGINEERING_BATCH_SIZE": ENGINEERING_BATCH_SIZE,
})

In [ ]:
all_prompt_texts = [BASE_TEMPLATE.format(name) for name in CIFAR10_CLASSES]
for template in ENSEMBLE_TEMPLATES:
    all_prompt_texts.extend(template.format(name) for name in CIFAR10_CLASSES)

vocab = {"<pad>": 0, "<unk>": 1}
for text in all_prompt_texts:
    for token in text.lower().replace(".", "").split():
        if token not in vocab:
            vocab[token] = len(vocab)

inv_vocab = {idx: token for token, idx in vocab.items()}

def encode_prompt(text: str, max_len: int = MAX_TEXT_LEN):
    tokens = text.lower().replace(".", "").split()
    token_ids = [vocab.get(token, vocab["<unk>"]) for token in tokens][:max_len]
    attention_mask = [1] * len(token_ids)
    while len(token_ids) < max_len:
        token_ids.append(vocab["<pad>"])
        attention_mask.append(0)
    return token_ids, attention_mask

base_prompts = [BASE_TEMPLATE.format(name) for name in CIFAR10_CLASSES]
class_token_ids = []
class_attention_masks = []
for prompt in base_prompts:
    token_ids, attention_mask = encode_prompt(prompt)
    class_token_ids.append(token_ids)
    class_attention_masks.append(attention_mask)

CLASS_TOKEN_IDS = torch.tensor(class_token_ids, dtype=torch.long)
CLASS_ATTENTION_MASKS = torch.tensor(class_attention_masks, dtype=torch.long)

def mean_pool(hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    weights = attention_mask.unsqueeze(-1).float()                        # (batch, seq, 1)
    pooled = (hidden_states * weights).sum(dim=1)                         # (batch, d_model)
    denom = weights.sum(dim=1).clamp(min=1.0)                             # (batch, 1)
    return pooled / denom

def contrastive_loss(image_features: torch.Tensor,
                     text_features: torch.Tensor,
                     logit_scale: torch.Tensor):
    image_features = F.normalize(image_features, dim=-1)
    text_features = F.normalize(text_features, dim=-1)

    logits = image_features @ text_features.T                             # (batch, batch)
    logits = logits * logit_scale.exp().clamp(max=100.0)

    labels = torch.arange(logits.size(0), device=logits.device)
    image_loss = F.cross_entropy(logits, labels)
    text_loss = F.cross_entropy(logits.T, labels)
    return 0.5 * (image_loss + text_loss), logits

print(f"Vocabulary size: {len(vocab)}")
print("Base prompt example:", base_prompts[3])
print("Encoded prompt example:", CLASS_TOKEN_IDS[3].tolist())

## Section iii：学习路径（L2：关键模块演示）

### 为什么这里选择 L2，而不是 L1 全量复现？
- 原论文的 CLIP 依赖超大规模图文对与强主干网络，完整复现不适合免费 Colab / CPU。
- 但 CLIP 的关键思想——**双编码器、共享嵌入空间、对比损失、prompt classifier**——完全可以稳定拆成独立模块演示。
- 因此这条学习路径的目标不是“复刻 OpenAI 训练规模”，而是“把最关键的数学对象都跑出来”。

### 组件 1：图像编码器（Image Encoder）

角色：把图像 $x$ 编成一个定长向量，再投射到共享嵌入空间。

$$I_f = f_{image}(x), \qquad I_e = W_i I_f$$

这里我们不复现论文中的 ResNet / ViT 主干，而是用一个足够小、足够稳定的 CNN 来演示：
- 输入：$(batch, 3, 32, 32)$
- 输出：$(batch, d_{model})$

直觉：图像编码器负责把局部像素模式压缩成语义向量；投射层负责把图像特征映射到“可以和文本直接点积”的空间。

In [ ]:
class TinyImageEncoder(nn.Module):
    """Tiny image encoder: small CNN + projection head."""
    def __init__(self, d_model: int = D_MODEL):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),   # (B, 3, 32, 32) -> (B, 32, 16, 16)
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # (B, 32, 16, 16) -> (B, 64, 8, 8)
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1),  # (B, 64, 8, 8) -> (B, 64, 4, 4)
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),                            # (B, 64, 4, 4) -> (B, 64, 1, 1)
        )
        self.projector = nn.Linear(64, d_model)                     # (B, 64) -> (B, d_model)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        x = self.features(images)
        x = x.flatten(1)                                            # (B, 64)
        x = self.projector(x)                                       # (B, d_model)
        return F.normalize(x, dim=-1)

image_encoder = TinyImageEncoder().to(device)
image_features = image_encoder(sample_images[:4].to(device))
print(f"Tiny image encoder output shape: {tuple(image_features.shape)}")

### 组件 2：文本编码器（Text Encoder）

角色：把 prompt 文本编码成向量，并投射到同一个共享空间。

$$T_f = f_{text}(t), \qquad T_e = W_t T_f$$

这里我们用一个教学版文本编码器：
- `Embedding`
- `Positional Embedding`
- `nn.MultiheadAttention(batch_first=True)`
- 前馈层 + 平均池化 + projector

输入 / 输出 shape：
- token ids：$(batch, seq)$
- hidden states：$(batch, seq, d_{model})$
- final text embedding：$(batch, d_{model})$

> 官方 PyTorch 文档里，`nn.MultiheadAttention(..., batch_first=True)` 的输入就是 `(batch, seq, embed_dim)`，这和我们这里的教学 shape 一致。

In [ ]:
class TinyTextEncoder(nn.Module):
    """Tiny text encoder: token embedding + self-attention + FFN + projection."""
    def __init__(self, vocab_size: int, d_model: int = D_MODEL, num_heads: int = NUM_HEADS, d_ff: int = D_FF):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.position_embedding = nn.Embedding(MAX_TEXT_LEN, d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.projector = nn.Linear(d_model, d_model)

    def forward(self, token_ids: torch.Tensor, attention_mask: torch.Tensor):
        batch_size, seq_len = token_ids.shape
        positions = torch.arange(seq_len, device=token_ids.device).unsqueeze(0)  # (1, seq)

        x = self.token_embedding(token_ids) + self.position_embedding(positions)  # (B, seq, d_model)
        key_padding_mask = attention_mask == 0                                     # (B, seq)

        attn_output, attn_weights = self.attn(
            x, x, x,
            key_padding_mask=key_padding_mask,
            need_weights=True,
            average_attn_weights=False,
        )
        x = self.norm1(x + attn_output)                                            # (B, seq, d_model)
        x = self.norm2(x + self.ffn(x))                                            # (B, seq, d_model)

        pooled = mean_pool(x, attention_mask)                                      # (B, d_model)
        pooled = self.projector(pooled)                                            # (B, d_model)
        return F.normalize(pooled, dim=-1), attn_weights

text_encoder = TinyTextEncoder(vocab_size=len(vocab)).to(device)
text_features, text_attn_weights = text_encoder(
    CLASS_TOKEN_IDS[:4].to(device),
    CLASS_ATTENTION_MASKS[:4].to(device),
)
print(f"Tiny text encoder output shape: {tuple(text_features.shape)}")
print(f"Attention weight shape: {tuple(text_attn_weights.shape)}")  # (B, heads, seq, seq)

### 组件 3：相似度矩阵、温度参数与对称损失

CLIP 最关键的不是“分类头”，而是一个 **batch 内所有图像与所有文本的相似度矩阵**：

$$\text{logits}_{ij} = \exp(s) \cdot \langle I_e^{(i)}, T_e^{(j)} \rangle$$

其中：
- 对角线：正确配对
- 非对角线：错误配对
- `logit_scale = exp(s)`：把余弦相似度放大 / 缩小

#### 训练 vs 推理的区别
- **训练**：一个 batch 内图像与文本一一配对，优化的是“对角线更大”。
- **推理**：先准备一组类别文本（prompt bank），再拿图像去和所有文本向量比相似度。

这也是 CLIP 和普通分类器最大的不同：它把“类别权重”换成了“文本特征”。

In [ ]:
class TinyCLIPDemo(nn.Module):
    def __init__(self, vocab_size: int):
        super().__init__()
        self.image_encoder = TinyImageEncoder(D_MODEL)
        self.text_encoder = TinyTextEncoder(vocab_size=vocab_size, d_model=D_MODEL, num_heads=NUM_HEADS, d_ff=D_FF)
        self.logit_scale = nn.Parameter(torch.tensor(np.log(1 / 0.07), dtype=torch.float32))

    def encode_image(self, images: torch.Tensor) -> torch.Tensor:
        return self.image_encoder(images)

    def encode_text(self, token_ids: torch.Tensor, attention_mask: torch.Tensor):
        text_features, attn_weights = self.text_encoder(token_ids, attention_mask)
        return text_features, attn_weights

    def forward_train(self, images: torch.Tensor, token_ids: torch.Tensor, attention_mask: torch.Tensor):
        image_features = self.encode_image(images)
        text_features, attn_weights = self.encode_text(token_ids, attention_mask)
        loss, logits = contrastive_loss(image_features, text_features, self.logit_scale)
        return loss, logits, attn_weights

tiny_clip = TinyCLIPDemo(vocab_size=len(vocab)).to(device)

demo_labels = sample_labels[:8]
demo_text_ids = CLASS_TOKEN_IDS[demo_labels].to(device)
demo_text_mask = CLASS_ATTENTION_MASKS[demo_labels].to(device)

demo_loss, demo_logits, demo_attn_weights = tiny_clip.forward_train(
    sample_images[:8].to(device),
    demo_text_ids,
    demo_text_mask,
)

print(f"Demo loss: {demo_loss.item():.4f}")
print(f"Similarity matrix shape: {tuple(demo_logits.shape)}")
print(f"Current logit scale: {tiny_clip.logit_scale.exp().item():.2f}")

plt.figure(figsize=(5, 4))
plt.imshow(demo_logits.detach().cpu().numpy(), cmap="viridis")
plt.colorbar()
plt.title("Batch Similarity Matrix")
plt.xlabel("Text in batch")
plt.ylabel("Image in batch")
plt.tight_layout()
plt.show()

### 训练 vs 推理的区别（Learning Path）

**训练阶段：**
- 文本输入是和 batch 图像标签对应的一组 prompt，如 `a photo of a cat`
- 每个 batch 都会构造一个 `N×N` 相似度矩阵
- 优化目标：拉高对角线，压低非对角线

**推理阶段：**
- 文本不再跟着 batch 样本动态变化，而是固定成一个“类别文本库”
- 图像只需要和这 10 个类别文本特征做相似度比较
- 预测类别 = 相似度最高的文本

下面先做一个 **小规模教学训练**，再做 zero-shot 推理。

In [ ]:
# 为了避免同类样本在同一 batch 内被错误当成负样本，
# 这里每个 step 只取每个类别各 1 张图像，形成 10-way paired batch。
unique_train_indices_by_class = {
    class_id: [idx for idx in train_indices if train_tensor_dataset.targets[idx] == class_id]
    for class_id in range(10)
}
steps_per_epoch = min(len(indices) for indices in unique_train_indices_by_class.values())

tiny_clip = TinyCLIPDemo(vocab_size=len(vocab)).to(device)
optimizer = torch.optim.Adam(tiny_clip.parameters(), lr=LEARNING_RATE)
learning_loss_history = []

for epoch in range(1, NUM_EPOCHS + 1):
    tiny_clip.train()
    running_loss = 0.0

    shuffled_indices_by_class = {
        class_id: np.random.permutation(indices)
        for class_id, indices in unique_train_indices_by_class.items()
    }

    for step in range(steps_per_epoch):
        batch_images = []
        batch_labels = []
        for class_id in range(10):
            dataset_index = int(shuffled_indices_by_class[class_id][step])
            image, label = train_tensor_dataset[dataset_index]
            batch_images.append(image)
            batch_labels.append(label)

        images = torch.stack(batch_images).to(device)                          # (10, 3, 32, 32)
        labels = torch.tensor(batch_labels, dtype=torch.long, device=device)   # (10,)
        token_ids = CLASS_TOKEN_IDS[labels].to(device)                         # (10, max_text_len)
        attention_mask = CLASS_ATTENTION_MASKS[labels].to(device)              # (10, max_text_len)

        loss, _, _ = tiny_clip.forward_train(images, token_ids, attention_mask)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / steps_per_epoch
    learning_loss_history.append(epoch_loss)
    print(f"Epoch {epoch}/{NUM_EPOCHS} - loss: {epoch_loss:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(range(1, NUM_EPOCHS + 1), learning_loss_history, marker="o")
plt.title("Tiny CLIP Demo Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Learning Path：推理（固定类别文本库）

下面把 10 个 CIFAR-10 类别文本先编码成一个固定的 text bank，
然后把每张图像编码成 image feature，再做点积相似度。

In [ ]:
@torch.no_grad()
def evaluate_tiny_clip(model: TinyCLIPDemo, loader: DataLoader):
    model.eval()
    text_bank, _ = model.encode_text(
        CLASS_TOKEN_IDS.to(device),
        CLASS_ATTENTION_MASKS.to(device),
    )                                                                    # (10, d_model)
    scale = model.logit_scale.exp().clamp(max=100.0)

    predictions = []
    labels_all = []
    for images, labels in loader:
        images = images.to(device)
        image_features = model.encode_image(images)                      # (batch, d_model)
        logits = (image_features @ text_bank.T) * scale                  # (batch, 10)
        preds = logits.argmax(dim=-1).cpu().tolist()
        predictions.extend(preds)
        labels_all.extend(labels.tolist())

    accuracy = (np.array(predictions) == np.array(labels_all)).mean()
    return accuracy, predictions, labels_all

learning_acc, learning_preds, eval_labels = evaluate_tiny_clip(tiny_clip, learning_test_loader)
print(f"Tiny CLIP demo zero-shot accuracy on the shared subset: {learning_acc * 100:.2f}%")
print("First 8 predictions:")
for idx in range(8):
    print(
        f"  #{idx:02d}  true={CIFAR10_CLASSES[eval_labels[idx]]:<10s}  "
        f"pred={CIFAR10_CLASSES[learning_preds[idx]]}"
    )

## Section iv：工程路径（E1：成熟库 + inference-only）

### 工程决策
- **工程路径形式**：E1（成熟工业库）
- **工程动作**：inference-only
- **原因**：`transformers` 已经提供成熟的 CLIP 处理器与预训练权重，直接做零样本推理最稳定、最符合工程使用场景。

这里使用：
- `AutoProcessor` 负责图像预处理 + 文本 tokenization
- `CLIPModel.from_pretrained("openai/clip-vit-base-patch32")`
- `get_image_features()` / `get_text_features()` 做批量特征提取

In [ ]:
CLIP_MODEL_ID = "openai/clip-vit-base-patch32"
processor = AutoProcessor.from_pretrained(CLIP_MODEL_ID)
engineering_model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(device)
engineering_model.eval()

num_params = sum(p.numel() for p in engineering_model.parameters())
print(f"Loaded model: {CLIP_MODEL_ID}")
print(f"Parameter count: {num_params:,}")
print(f"Logit scale: {engineering_model.logit_scale.exp().item():.2f}")

### 工程路径黑盒拆解（Black-box teardown）

#### `AutoProcessor(...)` 实际做了什么？
- 对图像：resize / center crop / normalize / 转成 `pixel_values`
- 对文本：tokenize / padding / attention mask
- 最终把“图像张量”和“文本张量”同时送进 CLIP

#### `model(...)` / `get_*_features()` 实际对应了什么？
- `get_image_features()`：学习路径里的 **图像编码器 + projector**
- `get_text_features()`：学习路径里的 **文本编码器 + projector**
- `outputs.logits_per_image`：学习路径里的 **相似度矩阵**
- `model.logit_scale.exp()`：学习路径里的 **温度参数**

| 学习路径实现 | 工程库内部对应 | 说明 |
|---|---|---|
| `TinyImageEncoder` | `CLIPModel.get_image_features()` | 图像编码器 + 视觉投射头 |
| `TinyTextEncoder` | `CLIPModel.get_text_features()` | 文本编码器 + 文本投射头 |
| `contrastive_loss()` 中的点积 | `outputs.logits_per_image` | 本质仍是归一化后点积 + 温度缩放 |
| 固定类别文本库 | `processor(text=labels, ...)` | 文本就是分类器权重 |
| 小规模教学训练 | 已在大规模图文对上离线完成 | 工程侧通常直接拿预训练模型推理 |

In [ ]:
@torch.no_grad()
def build_text_bank(model: CLIPModel, processor: AutoProcessor, prompts: List[str]) -> torch.Tensor:
    text_inputs = processor(text=prompts, return_tensors="pt", padding=True, truncation=True).to(device)
    text_features = model.get_text_features(**text_inputs)
    return F.normalize(text_features, dim=-1)

@torch.no_grad()
def build_ensemble_text_bank(model: CLIPModel, processor: AutoProcessor, class_names: List[str], templates: List[str]) -> torch.Tensor:
    class_features = []
    for class_name in class_names:
        prompts = [template.format(class_name) for template in templates]
        prompt_features = build_text_bank(model, processor, prompts)     # (num_templates, d_model)
        prompt_features = F.normalize(prompt_features.mean(dim=0, keepdim=True), dim=-1)
        class_features.append(prompt_features.squeeze(0))
    return torch.stack(class_features, dim=0)                            # (num_classes, d_model)

@torch.no_grad()
def evaluate_engineering_clip(raw_subset, text_bank: torch.Tensor, batch_size: int = ENGINEERING_BATCH_SIZE):
    predictions = []
    labels_all = []

    for start in range(0, len(raw_subset), batch_size):
        batch_items = [raw_subset[i] for i in range(start, min(start + batch_size, len(raw_subset)))]
        pil_images = [item[0] for item in batch_items]
        batch_labels = [item[1] for item in batch_items]

        image_inputs = processor(images=pil_images, return_tensors="pt").to(device)
        image_features = engineering_model.get_image_features(**image_inputs)
        image_features = F.normalize(image_features, dim=-1)

        logits = image_features @ text_bank.T
        logits = logits * engineering_model.logit_scale.exp()
        preds = logits.argmax(dim=-1).cpu().tolist()

        predictions.extend(preds)
        labels_all.extend(batch_labels)

    accuracy = (np.array(predictions) == np.array(labels_all)).mean()
    return accuracy, predictions, labels_all

# ---- Single example demo: official CLIP-style image-text similarity ----
demo_image, demo_label = engineering_test_subset[0]
demo_inputs = processor(text=base_prompts, images=demo_image, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    demo_outputs = engineering_model(**demo_inputs)
demo_probs = demo_outputs.logits_per_image.softmax(dim=-1).detach().cpu().numpy()[0]

top3 = np.argsort(demo_probs)[-3:][::-1]
print(f"Single example true label: {CIFAR10_CLASSES[demo_label]}")
for rank, class_id in enumerate(top3, start=1):
    print(f"Top {rank}: {base_prompts[class_id]}  prob={demo_probs[class_id]:.3f}")

# ---- Batch evaluation ----
base_text_bank = build_text_bank(engineering_model, processor, base_prompts)
ensemble_text_bank = build_ensemble_text_bank(engineering_model, processor, CIFAR10_CLASSES, ENSEMBLE_TEMPLATES)

engineering_acc_base, engineering_preds_base, engineering_labels = evaluate_engineering_clip(
    engineering_test_subset, base_text_bank
)
engineering_acc_ensemble, engineering_preds_ensemble, _ = evaluate_engineering_clip(
    engineering_test_subset, ensemble_text_bank
)

print(f"\nEngineering accuracy (base prompt):     {engineering_acc_base * 100:.2f}%")
print(f"Engineering accuracy (prompt ensemble): {engineering_acc_ensemble * 100:.2f}%")

### 工程路径的可调参数与权衡

| 因素 | 增大时 | 吞吐量 | 内存 | 速度 | 效果 |
|---|---|---|---|---|---|
| `batch_size` | 同时处理更多图像 | ↑ | ↑↑ | ↑（单样本平均更快） | 通常不变 |
| `num_prompts` | 每类模板更多 | ↓ | ↑ | ↓ | 可能 ↑ |
| 图像分辨率 | 输入更细 | ↓ | ↑↑ | ↓↓ | 可能 ↑ |
| 类别数 | 候选文本更多 | ↓ | ↑ | ↓ | 不直接提高，只更开放 |
| 预训练模型规模 | backbone 更大 | ↓ | ↑↑ | ↓ | 常常 ↑ |
| `device` CPU→GPU | 计算更并行 | ↑↑ | GPU-bound | ↑↑↑ | 通常不变 |

## Section v：学习路径 vs 工程路径对比（Mandatory）

| 对比维度 | 学习路径 | 工程路径 |
|---|---|---|
| 教育价值 | 直接看到 image/text encoder、相似度矩阵、温度参数、双向 CE | 直接看到工业 API 如何把这些步骤封装起来 |
| 代码量 | 更多，模块拆解细 | 更少，几行代码即可零样本推理 |
| 灵活性 | 高：可以自由替换 projector / loss / prompt 编码逻辑 | 中：受限于库接口和预训练结构 |
| 稳定性 | 中：教学代码更容易受超参数影响 | 高：预训练模型与处理器经过大量验证 |
| 工业适配度 | 适合教学、研究原型、机制分析 | 适合快速验证、检索、分类、生产原型 |
| 适用场景 | 面试准备<br>理解 CLIP 内部计算<br>做 ablation 小实验 | 零样本分类<br>批量推理<br>prompt engineering<br>快速 PoC |

## Section vi：训练 vs 推理差异（两条路径都要说清楚）

| 阶段 | 学习路径行为 | 工程路径行为 |
|---|---|---|
| 训练 | 小规模教学训练：一个 batch 内图像与 prompt 配对，优化对称对比损失 | 大规模预训练已离线完成；本 Notebook 不再训练 |
| 推理 | 先编码 10 个类别文本，图像与 text bank 做相似度比较 | `AutoProcessor` + `get_image_features()` / `get_text_features()`，本质同样是文本库匹配 |
| 温度参数 | `nn.Parameter(log(1/0.07))` 显式可见 | `engineering_model.logit_scale` 已包含在预训练模型里 |
| Prompt 作用 | 教学版只用基础模板 | 可直接切换多模板并做 prompt ensembling |

> 关键洞察：两条路径底层算法相同，只是抽象层次不同。

In [ ]:
print("First 10 predictions on the shared evaluation subset")
print("-" * 92)
print(f"{'idx':>3s} | {'true':<10s} | {'learning':<12s} | {'eng-base':<12s} | {'eng-ensemble':<12s}")
print("-" * 92)
for idx in range(10):
    true_name = CIFAR10_CLASSES[eval_labels[idx]]
    learning_name = CIFAR10_CLASSES[learning_preds[idx]]
    eng_base_name = CIFAR10_CLASSES[engineering_preds_base[idx]]
    eng_ens_name = CIFAR10_CLASSES[engineering_preds_ensemble[idx]]
    print(f"{idx:>3d} | {true_name:<10s} | {learning_name:<12s} | {eng_base_name:<12s} | {eng_ens_name:<12s}")

scores = [learning_acc * 100, engineering_acc_base * 100, engineering_acc_ensemble * 100]
labels = ["Tiny demo", "CLIP base prompt", "CLIP ensemble"]

plt.figure(figsize=(7, 4))
bars = plt.bar(labels, scores, color=["#4C72B0", "#55A868", "#C44E52"])
plt.ylabel("Accuracy (%)")
plt.title("Shared Evaluation Subset Accuracy")
plt.ylim(0, 100)
for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width() / 2, score + 1, f"{score:.1f}", ha="center")
plt.tight_layout()
plt.show()

## Section vii：结果、边界与适用条件

### 结果应该怎么读？
- **学习路径**的准确率不是论文结果，也不是工程上可直接使用的精度；它只是说明：哪怕把模型缩得很小，CLIP 的关键计算图仍然能跑通。
- **工程路径**的准确率更接近“真实可用”的 zero-shot 表现，但这里依然只是在一个很小的平衡子集上评估。

### 条件、范围与限制
- **适用条件**：你关心的是图像-文本对齐、prompt-based classification，而不是固定 softmax 分类头。
- **范围边界**：这里的结论只覆盖 CIFAR-10 小规模演示，不代表在细粒度识别、计数、复杂推理任务上也同样稳定。
- **已知限制**：CLIP 对 prompt wording 敏感；对抽象推理、计数、强 domain shift 场景并不理想。

### 什么时候用哪条路径？
- 如果你在准备面试 / 课程讲解：先走**学习路径**。
- 如果你要做零样本分类 / 原型验证 / 批量推理：直接走**工程路径**。
- 如果你要做研究：先用学习路径理解 loss 和 prompt，再在工程路径上做 prompt learning / fine-tuning / retrieval 扩展。

In [ ]:
# Appendix A: visualize text attention in the tiny text encoder
prompt_text = BASE_TEMPLATE.format("cat")
prompt_ids, prompt_mask = encode_prompt(prompt_text)
prompt_ids = torch.tensor([prompt_ids], dtype=torch.long, device=device)
prompt_mask = torch.tensor([prompt_mask], dtype=torch.long, device=device)

with torch.no_grad():
    _, attn_weights = tiny_clip.encode_text(prompt_ids, prompt_mask)

# attn_weights: (batch, heads, seq, seq)
head0 = attn_weights[0, 0].detach().cpu().numpy()
valid_tokens = [inv_vocab[idx] for idx in prompt_ids[0].detach().cpu().tolist() if idx != 0]
valid_len = len(valid_tokens)

plt.figure(figsize=(5, 4))
plt.imshow(head0[:valid_len, :valid_len], cmap="magma")
plt.xticks(range(valid_len), valid_tokens, rotation=45, ha="right")
plt.yticks(range(valid_len), valid_tokens)
plt.title("Tiny Text Attention (Head 0)")
plt.xlabel("Key tokens")
plt.ylabel("Query tokens")
plt.colorbar()
plt.tight_layout()
plt.show()

## Appendix B：面试与项目表达（Mandatory）

### 高频面试题

**Q1：CLIP 为什么能做 zero-shot classification？**
- 因为它学到的不是“固定 10 类分类头”，而是“图像和文本共用一个语义空间”。
- 推理时，把类别写成文本 prompt，本质上就是**临时生成分类器权重**。
- 所以新类别不一定要重新训练，只要你能写出合适的文本描述。

**Q2：为什么 CLIP 不直接生成文本，而是做图文匹配？**
- 生成完整文本比匹配正确配对更难、训练成本更高。
- CLIP 选择的是更直接的目标：在一个 batch 里找对图文配对。
- 这让它更适合做检索、零样本分类、跨模态对齐。

**Q3：温度参数 `logit_scale` 有什么作用？**
- 它控制相似度分布的“尖锐程度”。
- scale 更大，softmax 更尖锐，模型会更强调 hardest negatives。
- scale 太大也会让训练变得不稳定，所以通常会做裁剪或受控学习。

**Q4：为什么 CLIP 要做 L2 归一化？**
- 不归一化时，向量长度和方向会混在一起，点积不再只是“语义角度”的比较。
- 归一化后点积就是 cosine similarity，更适合做跨模态对齐。

**Q5：为什么 symmetric loss 要同时算 image→text 和 text→image？**
- 只优化单向，会让另一个方向的检索质量弱化。
- 双向 CE 可以同时约束“图找文”和“文找图”，让共享空间更对称。

**Q6：为什么 prompt wording 会影响 CLIP 表现？**
- 因为文本本身就是分类器的一部分。
- `dog` 和 `a photo of a dog` 在文本编码器里不是同一个分布。
- prompt 改写本质上是在改分类器权重，所以效果会明显变化。

**Q7：CLIP 的典型局限性是什么？**
- 对细粒度类别、计数、空间关系推理不强。
- 对领域外数据（如医学影像）不一定稳定。
- 对 prompt 很敏感，也会继承训练数据中的偏差。

**Q8：工程上为什么常做 prompt ensembling？**
- 单模板容易偏向某种语义措辞。
- 多模板平均可以降低 wording 偏差，让文本原型更稳。
- 这是零样本场景里成本最低、最常见的增强手段之一。

### 面试速答提纲

| # | 问题 | 一句话回答 |
|---|---|---|
| 1 | CLIP 本质在学什么？ | 学图像和文本的共享语义空间，而不是固定分类头。 |
| 2 | Zero-shot 分类为什么成立？ | 因为类别文本经过编码后就成了动态分类器。 |
| 3 | `logit_scale` 是干什么的？ | 控制相似度 softmax 的温度，影响区分度与训练稳定性。 |
| 4 | 为什么要做 L2 normalize？ | 把点积变成 cosine similarity，便于跨模态对齐。 |
| 5 | 为什么 prompt template 很重要？ | 因为文本本身就是分类器权重，措辞变了权重就变了。 |
| 6 | Prompt ensembling 为什么有效？ | 平均多个文本原型，降低单模板偏差。 |
| 7 | CLIP 和普通分类器最大区别？ | 普通分类器学固定标签头，CLIP 学开放词表文本对齐。 |

### 项目表达

> 如果面试官问“你做过什么相关项目”，可以这样组织回答：

- **学习深度**：我从零拆过 CLIP 的关键模块，包括双编码器、投射层、L2 归一化、对称对比损失和温度参数。
- **工程能力**：我用 `transformers` 的预训练 CLIP 做过 zero-shot 图像分类，并比较了单模板与 prompt ensembling。
- **对比思考**：我能说明教学版实现和工业版实现的区别：前者强调机制透明，后者强调稳定性与可部署性。
- **边界意识**：我知道 CLIP 在细粒度识别、计数、强领域迁移上有限，不能把 zero-shot 准确率泛化成“万能视觉理解”。

### 延伸阅读与对比

| 对比维度 | CLIP | 传统监督分类器 | 生成式多模态模型 |
|---|---|---|---|
| 核心思想 | 图文共享嵌入空间 | 固定标签 softmax 分类 | 条件生成 / 对话式理解 |
| 训练目标 | 对比学习 | 监督分类 | 生成损失 / 多任务损失 |
| 新类别适应 | 强，zero-shot | 弱，通常要重训 | 取决于指令与模型能力 |
| 典型场景 | 检索、零样本分类、跨模态对齐 | 封闭集分类 | 问答、描述、生成 |

### 进阶探索方向
- **Prompt learning**：从手写模板升级到可学习 prompt。
- **Retrieval**：把 CLIP 特征用于 image-text retrieval。
- **Domain adaptation**：在医学、遥感、电商等特定领域做轻量适配。
- **SigLIP / ALIGN / EVA-CLIP**：继续比较不同 vision-language 对齐目标的差异。